# ServiceNow Incident Data Extraction from Splunk

This notebook extracts ServiceNow Incident data from Splunk with the following features:
- **Incremental Loading**: Uses last ran date from tracking table (or default date)
- **Smart Chunking**: Splits large time ranges into manageable chunks
- **Retry Resilience**: Automatically retries failed queries with exponential backoff
- **Medallion Architecture**: Bronze (raw) → Silver (processed) layer pipeline
- **SLA Processing**: Response and Resolution SLA calculations
- **Incident Aggregation**: Handles multiple records per incident with proper state tracking

---

## Architecture Overview

```
┌─────────────┐     ┌─────────────────┐     ┌──────────────────────────┐
│   Splunk    │ ──► │  Bronze Layer   │ ──► │      Silver Layer        │
│  (Raw Data) │     │  (Raw Records)  │     │  (Processed + SLA Data)  │
└─────────────┘     └─────────────────┘     └──────────────────────────┘
                           │                            │
                    All raw events              Latest state per incident
                    from ServiceNow             + SLA breach calculations
```

## 1. Configuration & Imports

In [ ]:
import os
import sys
import time
import json
import logging
import requests
import urllib3
import pandas as pd
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional, Tuple
from pathlib import Path

# PySpark imports for Unity Catalog
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, coalesce,
    to_timestamp, unix_timestamp, datediff, expr,
    row_number, first, last, max as spark_max, min as spark_min,
    count, sum as spark_sum, avg, collect_list,
    from_json, get_json_object, explode, split,
    round as spark_round, floor, ceil,
    concat, concat_ws, trim, lower, upper,
    year, month, dayofmonth, hour, minute, date_format
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    TimestampType, LongType, DoubleType, BooleanType
)

# Disable SSL warnings if verify_ssl is False
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Get or create Spark session (assumes running in Databricks)
spark = SparkSession.builder.getOrCreate()

print("✅ Imports completed successfully")

## 2. Configuration Settings

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these settings as needed
# =============================================================================

# Splunk Connection Settings
SPLUNK_CONFIG = {
    'host': 'your-splunk-host.com',       # Splunk server hostname
    'port': 8089,                          # Management port
    'use_ssl': True,                       # Use HTTPS
    'verify_ssl': False,                   # Verify SSL certificates
    
    # Authentication (choose one method)
    'auth_type': 'token',                  # 'token' or 'basic'
    'token': 'your-splunk-token',          # For token auth
    # 'username': 'your-username',         # For basic auth
    # 'password': 'your-password',         # For basic auth
}

# ServiceNow Incident Query Settings
SERVICENOW_CONFIG = {
    'index': 'servicenow',                 # Splunk index containing ServiceNow data
    'sourcetype': 'servicenow:incident',   # Sourcetype for incident data
    'default_start_date': '2025-01-01',    # Default start date if no tracking data
    'max_results_per_chunk': 50000,        # Maximum results per query chunk
}

# Chunking Settings
CHUNK_CONFIG = {
    'chunk_size_hours': 24,                # Hours per chunk (adjust based on data volume)
    'min_chunk_size_hours': 1,             # Minimum chunk size for adaptive chunking
    'max_chunk_size_hours': 168,           # Maximum chunk size (7 days)
}

# Retry Settings
RETRY_CONFIG = {
    'max_retries': 3,                      # Maximum retry attempts
    'initial_delay_seconds': 2,            # Initial delay between retries
    'max_delay_seconds': 60,               # Maximum delay between retries
    'backoff_multiplier': 2,               # Exponential backoff multiplier
}

# Unity Catalog Settings - Medallion Architecture
UNITY_CATALOG_CONFIG = {
    'catalog': 'your_catalog',             # Unity Catalog name
    
    # Bronze Layer (Raw Data)
    'bronze_schema': 'servicenow_bronze',  # Bronze schema name
    'bronze_incidents_table': 'incidents_raw',  # Raw incidents table
    
    # Silver Layer (Processed Data)
    'silver_schema': 'servicenow_silver',  # Silver schema name
    'silver_incidents_table': 'incidents_processed',  # Processed incidents
    'silver_sla_table': 'incident_sla_metrics',  # SLA metrics table
    
    # Tracking
    'tracking_schema': 'servicenow_bronze',  # Schema for tracking table
    'tracking_table': 'extraction_tracking',  # Tracking table name
}

# SLA Configuration (in minutes)
SLA_CONFIG = {
    # Response SLA by Priority (time to first response in minutes)
    'response_sla': {
        '1': 15,      # Priority 1 - Critical: 15 minutes
        '2': 30,      # Priority 2 - High: 30 minutes
        '3': 60,      # Priority 3 - Medium: 1 hour
        '4': 240,     # Priority 4 - Low: 4 hours
        '5': 480,     # Priority 5 - Planning: 8 hours
    },
    
    # Resolution SLA by Priority (time to resolution in minutes)
    'resolution_sla': {
        '1': 240,     # Priority 1 - Critical: 4 hours
        '2': 480,     # Priority 2 - High: 8 hours
        '3': 1440,    # Priority 3 - Medium: 24 hours
        '4': 2880,    # Priority 4 - Low: 48 hours
        '5': 10080,   # Priority 5 - Planning: 7 days
    },
    
    # Business hours configuration (for SLA calculations)
    'business_hours': {
        'start_hour': 8,   # 8 AM
        'end_hour': 18,    # 6 PM
        'working_days': [0, 1, 2, 3, 4],  # Monday=0 to Friday=4
    },
    
    # Use business hours for SLA calculation (True) or 24x7 (False)
    'use_business_hours': False,
}

# Tracking Table Settings
TRACKING_CONFIG = {
    'job_name': 'servicenow_incident_extraction',  # Unique job identifier
}

# Field Mapping - Map Splunk fields to standardized names
FIELD_MAPPING = {
    # Raw field name -> Standardized name
    'number': 'incident_number',
    'sys_id': 'incident_sys_id',
    'sys_created_on': 'created_at',
    'sys_updated_on': 'updated_at',
    'opened_at': 'opened_at',
    'resolved_at': 'resolved_at',
    'closed_at': 'closed_at',
    'response_time': 'first_response_at',
    'work_start': 'work_start_at',
}

print("✅ Configuration loaded")

## 3. Core Classes & Functions

In [ ]:
class UnityCatalogTrackingTable:
    """
    Manages the tracking table in Unity Catalog for incremental data loading.
    Stores last successful run timestamps for each job.
    """
    
    def __init__(self, catalog: str, schema: str, table: str):
        self.catalog = catalog
        self.schema = schema
        self.table = table
        self.full_table_name = f"{catalog}.{schema}.{table}"
        self._ensure_tracking_table()
    
    def _ensure_tracking_table(self):
        """Create tracking table in Unity Catalog if it doesn't exist."""
        # Create schema if not exists
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{self.schema}")
        
        # Create tracking table if not exists
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.full_table_name} (
                job_name STRING,
                last_run_date TIMESTAMP,
                records_extracted LONG,
                status STRING,
                updated_at TIMESTAMP
            )
            USING DELTA
        """)
        logger.info(f"Tracking table ready: {self.full_table_name}")
    
    def get_last_run_date(self, job_name: str, default_date: str) -> datetime:
        """
        Get the last successful run date for a job from Unity Catalog.
        
        Args:
            job_name: Unique identifier for the extraction job
            default_date: Default date to use if no previous run exists (YYYY-MM-DD)
            
        Returns:
            datetime: Last run datetime or default
        """
        try:
            result_df = spark.sql(f"""
                SELECT last_run_date 
                FROM {self.full_table_name}
                WHERE job_name = '{job_name}' AND status = 'SUCCESS'
                ORDER BY last_run_date DESC
                LIMIT 1
            """)
            
            if result_df.count() > 0:
                last_run_dt = result_df.first()['last_run_date']
                logger.info(f"📅 Found last run date: {last_run_dt}")
                return last_run_dt
            else:
                default_dt = pd.to_datetime(default_date).to_pydatetime()
                logger.info(f"📅 No previous run found. Using default date: {default_dt}")
                return default_dt
                
        except Exception as e:
            logger.warning(f"Error reading tracking table: {e}. Using default date.")
            return pd.to_datetime(default_date).to_pydatetime()
    
    def update_tracking(self, job_name: str, last_run_date: datetime, 
                        records_extracted: int, status: str = 'SUCCESS'):
        """
        Update the tracking table in Unity Catalog with the latest run information.
        
        Args:
            job_name: Unique identifier for the extraction job
            last_run_date: The end date of the extraction window
            records_extracted: Number of records extracted
            status: Run status ('SUCCESS', 'FAILED', 'PARTIAL')
        """
        try:
            # Create a DataFrame with the new record
            new_record = [(
                job_name,
                last_run_date,
                records_extracted,
                status,
                datetime.now()
            )]
            
            schema = StructType([
                StructField("job_name", StringType(), False),
                StructField("last_run_date", TimestampType(), False),
                StructField("records_extracted", LongType(), False),
                StructField("status", StringType(), False),
                StructField("updated_at", TimestampType(), False)
            ])
            
            new_df = spark.createDataFrame(new_record, schema)
            
            # Append to tracking table
            new_df.write.mode("append").saveAsTable(self.full_table_name)
            logger.info(f"✅ Tracking table updated: {records_extracted} records, status={status}")
            
        except Exception as e:
            logger.error(f"Failed to update tracking table: {e}")
            raise


print("✅ UnityCatalogTrackingTable class defined")

In [ ]:
class TimeChunker:
    """
    Splits a time range into manageable chunks for processing.
    Supports adaptive chunking based on data volume.
    """
    
    def __init__(self, chunk_size_hours: int = 24, 
                 min_chunk_hours: int = 1, 
                 max_chunk_hours: int = 168):
        self.chunk_size_hours = chunk_size_hours
        self.min_chunk_hours = min_chunk_hours
        self.max_chunk_hours = max_chunk_hours
    
    def generate_chunks(self, start_date: datetime, end_date: datetime) -> List[Tuple[datetime, datetime]]:
        """
        Generate time chunks between start and end dates.
        
        Args:
            start_date: Start of the time range
            end_date: End of the time range
            
        Returns:
            List of (chunk_start, chunk_end) tuples
        """
        chunks = []
        current = start_date
        chunk_delta = timedelta(hours=self.chunk_size_hours)
        
        while current < end_date:
            chunk_end = min(current + chunk_delta, end_date)
            chunks.append((current, chunk_end))
            current = chunk_end
        
        logger.info(f"📊 Generated {len(chunks)} time chunks from {start_date} to {end_date}")
        return chunks
    
    def split_chunk(self, chunk_start: datetime, chunk_end: datetime) -> List[Tuple[datetime, datetime]]:
        """
        Split a chunk in half for adaptive processing when a chunk is too large.
        
        Args:
            chunk_start: Start of the chunk to split
            chunk_end: End of the chunk to split
            
        Returns:
            List of two smaller chunks
        """
        duration = chunk_end - chunk_start
        if duration.total_seconds() / 3600 <= self.min_chunk_hours:
            logger.warning(f"⚠️ Chunk already at minimum size, cannot split further")
            return [(chunk_start, chunk_end)]
        
        mid_point = chunk_start + (duration / 2)
        logger.info(f"🔀 Splitting chunk: {chunk_start} - {chunk_end} into two parts")
        return [(chunk_start, mid_point), (mid_point, chunk_end)]


print("✅ TimeChunker class defined")

In [ ]:
class RetryHandler:
    """
    Handles retry logic with exponential backoff.
    """
    
    def __init__(self, max_retries: int = 3, 
                 initial_delay: float = 2, 
                 max_delay: float = 60,
                 backoff_multiplier: float = 2):
        self.max_retries = max_retries
        self.initial_delay = initial_delay
        self.max_delay = max_delay
        self.backoff_multiplier = backoff_multiplier
    
    def execute_with_retry(self, func, *args, **kwargs) -> Any:
        """
        Execute a function with retry logic.
        
        Args:
            func: Function to execute
            *args: Positional arguments for the function
            **kwargs: Keyword arguments for the function
            
        Returns:
            Result of the function if successful
            
        Raises:
            Exception: If all retries fail
        """
        last_exception = None
        delay = self.initial_delay
        
        for attempt in range(self.max_retries + 1):
            try:
                if attempt > 0:
                    logger.info(f"🔄 Retry attempt {attempt}/{self.max_retries}")
                return func(*args, **kwargs)
                
            except Exception as e:
                last_exception = e
                
                if attempt < self.max_retries:
                    logger.warning(f"⚠️ Attempt {attempt + 1} failed: {str(e)}")
                    logger.info(f"⏳ Waiting {delay:.1f} seconds before retry...")
                    time.sleep(delay)
                    delay = min(delay * self.backoff_multiplier, self.max_delay)
                else:
                    logger.error(f"❌ All {self.max_retries + 1} attempts failed")
        
        raise last_exception


print("✅ RetryHandler class defined")

In [ ]:
class SplunkIncidentExtractor:
    """
    Main class for extracting ServiceNow Incident data from Splunk using REST API.
    Combines time chunking with retry resilience.
    """
    
    def __init__(self, splunk_config: dict, servicenow_config: dict,
                 chunk_config: dict, retry_config: dict):
        self.splunk_config = splunk_config
        self.servicenow_config = servicenow_config
        self.session = requests.Session()
        self.base_url = self._build_base_url()
        self.retry_handler = RetryHandler(**retry_config)
        self.chunker = TimeChunker(
            chunk_size_hours=chunk_config['chunk_size_hours'],
            min_chunk_hours=chunk_config['min_chunk_size_hours'],
            max_chunk_hours=chunk_config['max_chunk_size_hours']
        )
        self.is_connected = False
    
    def _build_base_url(self) -> str:
        """Build the base URL for Splunk REST API."""
        scheme = 'https' if self.splunk_config['use_ssl'] else 'http'
        return f"{scheme}://{self.splunk_config['host']}:{self.splunk_config['port']}"
    
    def _get_auth_headers(self) -> Dict[str, str]:
        """Get authentication headers based on auth type."""
        if self.splunk_config['auth_type'] == 'token':
            return {'Authorization': f"Bearer {self.splunk_config['token']}"}
        return {}
    
    def _get_auth(self) -> Optional[Tuple[str, str]]:
        """Get basic auth tuple if using basic authentication."""
        if self.splunk_config['auth_type'] == 'basic':
            return (self.splunk_config['username'], self.splunk_config['password'])
        return None
    
    def connect(self) -> bool:
        """
        Verify connection to Splunk REST API with retry logic.
        
        Returns:
            bool: True if connection successful
        """
        def _connect():
            url = f"{self.base_url}/services/server/info"
            response = self.session.get(
                url,
                headers=self._get_auth_headers(),
                auth=self._get_auth(),
                verify=self.splunk_config['verify_ssl'],
                params={'output_mode': 'json'}
            )
            response.raise_for_status()
            return True
        
        try:
            self.retry_handler.execute_with_retry(_connect)
            self.is_connected = True
            logger.info(f"✅ Connected to Splunk REST API: {self.base_url}")
            return True
        except Exception as e:
            logger.error(f"❌ Failed to connect to Splunk: {e}")
            return False
    
    def build_query(self, start_time: datetime, end_time: datetime) -> str:
        """
        Build the SPL query for ServiceNow incidents.
        
        Args:
            start_time: Start of the time range
            end_time: End of the time range
            
        Returns:
            str: SPL query string
        """
        start_str = start_time.strftime('%Y-%m-%dT%H:%M:%S')
        end_str = end_time.strftime('%Y-%m-%dT%H:%M:%S')
        
        query = f"""
search index={self.servicenow_config['index']} 
       sourcetype="{self.servicenow_config['sourcetype']}"
       earliest="{start_str}"
       latest="{end_str}"
| fields _time, number, sys_id, short_description, description, 
         priority, urgency, impact, state, category, subcategory,
         assignment_group, assigned_to, caller_id, opened_by,
         opened_at, resolved_at, closed_at, close_code, close_notes,
         cmdb_ci, business_service, location, company
| sort 0 _time
        """.strip()
        
        return query
    
    def _create_search_job(self, query: str, max_results: int) -> str:
        """Create a search job via REST API."""
        url = f"{self.base_url}/services/search/jobs"
        data = {
            'search': query,
            'output_mode': 'json',
            'count': max_results,
            'exec_mode': 'normal'
        }
        
        response = self.session.post(
            url,
            headers=self._get_auth_headers(),
            auth=self._get_auth(),
            data=data,
            verify=self.splunk_config['verify_ssl']
        )
        response.raise_for_status()
        
        result = response.json()
        return result['sid']
    
    def _wait_for_job(self, sid: str, poll_interval: float = 1.0) -> bool:
        """Wait for a search job to complete."""
        url = f"{self.base_url}/services/search/jobs/{sid}"
        
        while True:
            response = self.session.get(
                url,
                headers=self._get_auth_headers(),
                auth=self._get_auth(),
                params={'output_mode': 'json'},
                verify=self.splunk_config['verify_ssl']
            )
            response.raise_for_status()
            
            job_info = response.json()
            dispatch_state = job_info['entry'][0]['content']['dispatchState']
            
            if dispatch_state == 'DONE':
                return True
            elif dispatch_state == 'FAILED':
                raise Exception(f"Search job {sid} failed")
            
            time.sleep(poll_interval)
    
    def _get_job_results(self, sid: str, max_results: int) -> List[Dict[str, Any]]:
        """Get results from a completed search job."""
        url = f"{self.base_url}/services/search/jobs/{sid}/results"
        all_results = []
        offset = 0
        batch_size = min(10000, max_results)
        
        while offset < max_results:
            response = self.session.get(
                url,
                headers=self._get_auth_headers(),
                auth=self._get_auth(),
                params={
                    'output_mode': 'json',
                    'count': batch_size,
                    'offset': offset
                },
                verify=self.splunk_config['verify_ssl']
            )
            response.raise_for_status()
            
            data = response.json()
            results = data.get('results', [])
            
            if not results:
                break
            
            all_results.extend(results)
            offset += len(results)
            
            if len(results) < batch_size:
                break
        
        return all_results
    
    def _cancel_job(self, sid: str):
        """Cancel/delete a search job."""
        try:
            url = f"{self.base_url}/services/search/jobs/{sid}/control"
            self.session.post(
                url,
                headers=self._get_auth_headers(),
                auth=self._get_auth(),
                data={'action': 'cancel'},
                verify=self.splunk_config['verify_ssl']
            )
        except Exception:
            pass
    
    def execute_query(self, query: str, max_results: int = 50000) -> List[Dict[str, Any]]:
        """Execute a Splunk query via REST API and return results."""
        def _execute():
            sid = self._create_search_job(query, max_results)
            logger.debug(f"Created search job: {sid}")
            
            try:
                self._wait_for_job(sid)
                results = self._get_job_results(sid, max_results)
                return results
            finally:
                self._cancel_job(sid)
        
        return self.retry_handler.execute_with_retry(_execute)
    
    def extract_chunk(self, chunk_start: datetime, chunk_end: datetime) -> List[Dict[str, Any]]:
        """Extract data for a single time chunk with adaptive splitting."""
        max_results = self.servicenow_config['max_results_per_chunk']
        
        try:
            query = self.build_query(chunk_start, chunk_end)
            results = self.execute_query(query, max_results)
            
            logger.info(f"📦 Chunk {chunk_start} to {chunk_end}: {len(results)} records")
            
            if len(results) >= max_results:
                logger.warning(f"⚠️ Chunk may have more data, splitting for complete extraction")
                smaller_chunks = self.chunker.split_chunk(chunk_start, chunk_end)
                
                if len(smaller_chunks) > 1:
                    results = []
                    for sub_start, sub_end in smaller_chunks:
                        sub_results = self.extract_chunk(sub_start, sub_end)
                        results.extend(sub_results)
            
            return results
            
        except Exception as e:
            logger.error(f"❌ Failed to extract chunk {chunk_start} to {chunk_end}: {e}")
            raise
    
    def extract_all(self, start_date: datetime, end_date: datetime) -> pd.DataFrame:
        """Extract all data between start and end dates using smart chunking."""
        logger.info(f"🚀 Starting extraction from {start_date} to {end_date}")
        
        chunks = self.chunker.generate_chunks(start_date, end_date)
        all_results = []
        failed_chunks = []
        
        for i, (chunk_start, chunk_end) in enumerate(chunks, 1):
            logger.info(f"\n📍 Processing chunk {i}/{len(chunks)}")
            
            try:
                results = self.extract_chunk(chunk_start, chunk_end)
                all_results.extend(results)
                logger.info(f"✅ Chunk {i} complete: {len(results)} records (Total: {len(all_results)})")
                
            except Exception as e:
                logger.error(f"❌ Chunk {i} failed permanently: {e}")
                failed_chunks.append((chunk_start, chunk_end, str(e)))
        
        if failed_chunks:
            logger.warning(f"\n⚠️ {len(failed_chunks)} chunks failed:")
            for start, end, error in failed_chunks:
                logger.warning(f"  - {start} to {end}: {error}")
        
        logger.info(f"\n🎉 Extraction complete: {len(all_results)} total records")
        
        if all_results:
            return pd.DataFrame(all_results)
        else:
            return pd.DataFrame()


print("✅ SplunkIncidentExtractor class defined")

In [ ]:
class SLAProcessor:
    """
    Processes ServiceNow incident data to calculate SLA metrics.
    Handles Response SLA and Resolution SLA calculations.
    """
    
    def __init__(self, sla_config: dict):
        self.response_sla = sla_config['response_sla']
        self.resolution_sla = sla_config['resolution_sla']
        self.business_hours = sla_config['business_hours']
        self.use_business_hours = sla_config['use_business_hours']
    
    def get_response_sla_minutes(self, priority: str) -> int:
        """Get response SLA target in minutes for a priority."""
        return self.response_sla.get(str(priority), self.response_sla.get('3', 60))
    
    def get_resolution_sla_minutes(self, priority: str) -> int:
        """Get resolution SLA target in minutes for a priority."""
        return self.resolution_sla.get(str(priority), self.resolution_sla.get('3', 1440))
    
    def create_sla_lookup_df(self):
        """Create a Spark DataFrame with SLA targets for joining."""
        sla_data = []
        for priority in ['1', '2', '3', '4', '5']:
            sla_data.append((
                priority,
                self.get_response_sla_minutes(priority),
                self.get_resolution_sla_minutes(priority)
            ))
        
        schema = StructType([
            StructField("priority", StringType(), False),
            StructField("response_sla_target_minutes", IntegerType(), False),
            StructField("resolution_sla_target_minutes", IntegerType(), False)
        ])
        
        return spark.createDataFrame(sla_data, schema)
    
    def process_silver_layer(self, bronze_df):
        """
        Process bronze data into silver layer with SLA calculations.
        
        Args:
            bronze_df: Raw incident data from bronze layer
            
        Returns:
            Tuple of (processed_incidents_df, sla_metrics_df)
        """
        # Create SLA lookup
        sla_lookup_df = self.create_sla_lookup_df()
        
        # Window for getting latest record per incident
        incident_window = Window.partitionBy("number").orderBy(col("_extracted_at").desc())
        
        # === PROCESSED INCIDENTS (Latest State per Incident) ===
        processed_df = bronze_df \
            .withColumn("_row_num", row_number().over(incident_window)) \
            .filter(col("_row_num") == 1) \
            .drop("_row_num")
        
        # Parse timestamp fields (handle various ServiceNow date formats)
        timestamp_cols = ['opened_at', 'resolved_at', 'closed_at', 'sys_created_on', 
                         'sys_updated_on', 'response_time', 'work_start', 'first_response_time']
        
        for ts_col in timestamp_cols:
            if ts_col in processed_df.columns:
                processed_df = processed_df.withColumn(
                    f"{ts_col}_ts",
                    coalesce(
                        to_timestamp(col(ts_col), "yyyy-MM-dd HH:mm:ss"),
                        to_timestamp(col(ts_col), "yyyy-MM-dd'T'HH:mm:ss"),
                        to_timestamp(col(ts_col))
                    )
                )
        
        # Rename priority column for join if needed
        priority_col = "priority" if "priority" in processed_df.columns else "urgency"
        
        # Join with SLA targets
        processed_df = processed_df.join(
            sla_lookup_df,
            col(priority_col) == sla_lookup_df["priority"],
            "left"
        ).drop(sla_lookup_df["priority"])
        
        # Determine first response timestamp
        first_response_col = None
        for potential_col in ['first_response_time_ts', 'response_time_ts', 'work_start_ts']:
            if potential_col in processed_df.columns:
                first_response_col = potential_col
                break
        
        # Calculate Response Time in Minutes
        if first_response_col and 'opened_at_ts' in processed_df.columns:
            processed_df = processed_df.withColumn(
                "response_time_minutes",
                when(
                    col(first_response_col).isNotNull() & col("opened_at_ts").isNotNull(),
                    spark_round((unix_timestamp(col(first_response_col)) - unix_timestamp(col("opened_at_ts"))) / 60, 2)
                ).otherwise(lit(None))
            )
        else:
            processed_df = processed_df.withColumn("response_time_minutes", lit(None))
        
        # Calculate Resolution Time in Minutes
        if 'resolved_at_ts' in processed_df.columns and 'opened_at_ts' in processed_df.columns:
            processed_df = processed_df.withColumn(
                "resolution_time_minutes",
                when(
                    col("resolved_at_ts").isNotNull() & col("opened_at_ts").isNotNull(),
                    spark_round((unix_timestamp(col("resolved_at_ts")) - unix_timestamp(col("opened_at_ts"))) / 60, 2)
                ).otherwise(lit(None))
            )
        else:
            processed_df = processed_df.withColumn("resolution_time_minutes", lit(None))
        
        # Calculate SLA Breach flags
        processed_df = processed_df \
            .withColumn(
                "response_sla_breached",
                when(
                    col("response_time_minutes").isNotNull() & col("response_sla_target_minutes").isNotNull(),
                    col("response_time_minutes") > col("response_sla_target_minutes")
                ).otherwise(lit(None))
            ) \
            .withColumn(
                "resolution_sla_breached",
                when(
                    col("resolution_time_minutes").isNotNull() & col("resolution_sla_target_minutes").isNotNull(),
                    col("resolution_time_minutes") > col("resolution_sla_target_minutes")
                ).otherwise(lit(None))
            )
        
        # Calculate remaining SLA time for open incidents
        processed_df = processed_df \
            .withColumn(
                "response_sla_remaining_minutes",
                when(
                    col("response_time_minutes").isNull() & col("opened_at_ts").isNotNull() & col("response_sla_target_minutes").isNotNull(),
                    spark_round(col("response_sla_target_minutes") - 
                    (unix_timestamp(current_timestamp()) - unix_timestamp(col("opened_at_ts"))) / 60, 2)
                ).when(
                    col("response_time_minutes").isNotNull() & col("response_sla_target_minutes").isNotNull(),
                    spark_round(col("response_sla_target_minutes") - col("response_time_minutes"), 2)
                ).otherwise(lit(None))
            ) \
            .withColumn(
                "resolution_sla_remaining_minutes",
                when(
                    col("resolution_time_minutes").isNull() & col("opened_at_ts").isNotNull() & col("resolution_sla_target_minutes").isNotNull(),
                    spark_round(col("resolution_sla_target_minutes") - 
                    (unix_timestamp(current_timestamp()) - unix_timestamp(col("opened_at_ts"))) / 60, 2)
                ).when(
                    col("resolution_time_minutes").isNotNull() & col("resolution_sla_target_minutes").isNotNull(),
                    spark_round(col("resolution_sla_target_minutes") - col("resolution_time_minutes"), 2)
                ).otherwise(lit(None))
            ) \
            .withColumn("_processed_at", current_timestamp())
        
        # === SLA METRICS AGGREGATION ===
        sla_metrics_df = processed_df \
            .groupBy(priority_col) \
            .agg(
                count("*").alias("total_incidents"),
                spark_sum(when(col("response_sla_breached") == True, 1).otherwise(0)).alias("response_sla_breaches"),
                spark_sum(when(col("resolution_sla_breached") == True, 1).otherwise(0)).alias("resolution_sla_breaches"),
                spark_round(avg(col("response_time_minutes")), 2).alias("avg_response_time_minutes"),
                spark_round(avg(col("resolution_time_minutes")), 2).alias("avg_resolution_time_minutes"),
                spark_min(col("response_time_minutes")).alias("min_response_time_minutes"),
                spark_max(col("response_time_minutes")).alias("max_response_time_minutes"),
                spark_min(col("resolution_time_minutes")).alias("min_resolution_time_minutes"),
                spark_max(col("resolution_time_minutes")).alias("max_resolution_time_minutes"),
                count(when(col("state").isin("Resolved", "6"), 1)).alias("resolved_count"),
                count(when(col("state").isin("Closed", "7"), 1)).alias("closed_count"),
                count(when(col("state").isin("New", "In Progress", "On Hold", "1", "2", "3"), 1)).alias("open_count"),
                first(col("response_sla_target_minutes")).alias("response_sla_target_minutes"),
                first(col("resolution_sla_target_minutes")).alias("resolution_sla_target_minutes")
            ) \
            .withColumn(
                "response_sla_compliance_pct",
                when(
                    col("total_incidents") > 0,
                    spark_round((col("total_incidents") - col("response_sla_breaches")) / col("total_incidents") * 100, 2)
                ).otherwise(lit(100.0))
            ) \
            .withColumn(
                "resolution_sla_compliance_pct",
                when(
                    col("total_incidents") > 0,
                    spark_round((col("total_incidents") - col("resolution_sla_breaches")) / col("total_incidents") * 100, 2)
                ).otherwise(lit(100.0))
            ) \
            .withColumn("_metrics_calculated_at", current_timestamp())
        
        return processed_df, sla_metrics_df


print("✅ SLAProcessor class defined")

In [ ]:
def save_to_bronze_layer(df, unity_config: dict, partition_cols: list = None):
    """
    Save raw Splunk data to Bronze layer (Unity Catalog).
    
    Args:
        df: PySpark DataFrame with raw incident data
        unity_config: Unity Catalog configuration
        partition_cols: Optional columns to partition by
        
    Returns:
        Number of records saved
    """
    catalog = unity_config['catalog']
    schema = unity_config['bronze_schema']
    table = unity_config['bronze_incidents_table']
    full_table_name = f"{catalog}.{schema}.{table}"
    
    # Add bronze layer metadata
    bronze_df = df \
        .withColumn("_bronze_loaded_at", current_timestamp()) \
        .withColumn("_source_system", lit("splunk"))
    
    # Create schema if not exists
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    
    # Write to Bronze table
    write_mode = "append"  # Bronze layer always appends (raw history)
    
    if partition_cols:
        bronze_df.write \
            .mode(write_mode) \
            .partitionBy(*partition_cols) \
            .format("delta") \
            .saveAsTable(full_table_name)
    else:
        bronze_df.write \
            .mode(write_mode) \
            .format("delta") \
            .saveAsTable(full_table_name)
    
    record_count = bronze_df.count()
    print(f"✅ Saved {record_count} records to Bronze layer: {full_table_name}")
    
    return record_count


def process_and_save_silver_layer(bronze_df, sla_processor: SLAProcessor, unity_config: dict):
    """
    Process Bronze data and save to Silver layer with SLA calculations.
    
    Args:
        bronze_df: Raw data from Bronze layer
        sla_processor: SLAProcessor instance
        unity_config: Unity Catalog configuration
        
    Returns:
        Tuple of (incidents_count, sla_metrics_count)
    """
    catalog = unity_config['catalog']
    schema = unity_config['silver_schema']
    incidents_table = unity_config['silver_incidents_table']
    sla_table = unity_config['silver_sla_table']
    
    full_incidents_table = f"{catalog}.{schema}.{incidents_table}"
    full_sla_table = f"{catalog}.{schema}.{sla_table}"
    
    # Create schema if not exists
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    
    # Process data through SLA calculations
    processed_incidents_df, sla_metrics_df = sla_processor.process_silver_layer(bronze_df)
    
    # === Save Processed Incidents (Merge on incident number) ===
    if spark.catalog.tableExists(full_incidents_table):
        # Use MERGE for upsert
        processed_incidents_df.createOrReplaceTempView("updates")
        
        # Build merge statement dynamically based on available columns
        update_cols = [c for c in processed_incidents_df.columns if not c.startswith('_')]
        set_clause = ", ".join([f"target.{c} = source.{c}" for c in update_cols])
        insert_cols = ", ".join(processed_incidents_df.columns)
        insert_vals = ", ".join([f"source.{c}" for c in processed_incidents_df.columns])
        
        merge_sql = f"""
        MERGE INTO {full_incidents_table} AS target
        USING updates AS source
        ON target.number = source.number
        WHEN MATCHED THEN UPDATE SET {set_clause}, target._processed_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
        """
        spark.sql(merge_sql)
        incidents_count = processed_incidents_df.count()
    else:
        # First time - create table
        processed_incidents_df.write \
            .mode("overwrite") \
            .format("delta") \
            .saveAsTable(full_incidents_table)
        incidents_count = processed_incidents_df.count()
    
    print(f"✅ Saved {incidents_count} processed incidents to Silver layer: {full_incidents_table}")
    
    # === Save SLA Metrics (Overwrite each run) ===
    sla_metrics_df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(full_sla_table)
    
    sla_count = sla_metrics_df.count()
    print(f"✅ Saved {sla_count} SLA metrics to Silver layer: {full_sla_table}")
    
    return incidents_count, sla_count


def load_bronze_data_for_processing(unity_config: dict, since_timestamp=None):
    """
    Load Bronze data for Silver layer processing.
    
    Args:
        unity_config: Unity Catalog configuration
        since_timestamp: Optional timestamp to filter records loaded after this time
        
    Returns:
        PySpark DataFrame
    """
    catalog = unity_config['catalog']
    schema = unity_config['bronze_schema']
    table = unity_config['bronze_incidents_table']
    full_table_name = f"{catalog}.{schema}.{table}"
    
    if not spark.catalog.tableExists(full_table_name):
        print(f"⚠️ Bronze table {full_table_name} does not exist")
        return None
    
    bronze_df = spark.table(full_table_name)
    
    if since_timestamp:
        bronze_df = bronze_df.filter(col("_bronze_loaded_at") >= since_timestamp)
    
    record_count = bronze_df.count()
    print(f"📊 Loaded {record_count} records from Bronze layer")
    
    return bronze_df


print("✅ Bronze and Silver layer functions defined")

## 4. Initialize Components

In [ ]:
# Initialize Unity Catalog tracking table
tracking = UnityCatalogTrackingTable(
    catalog=UNITY_CATALOG_CONFIG['catalog'],
    schema=UNITY_CATALOG_CONFIG['tracking_schema'],
    table=UNITY_CATALOG_CONFIG['tracking_table']
)

# Initialize SLA Processor
sla_processor = SLAProcessor(SLA_CONFIG)

# Get last run date or use default
last_run_date = tracking.get_last_run_date(
    job_name=TRACKING_CONFIG['job_name'],
    default_date=SERVICENOW_CONFIG['default_start_date']
)

# Set end date to current time
end_date = datetime.now()

print(f"\n📊 Extraction Window:")
print(f"   Start: {last_run_date}")
print(f"   End:   {end_date}")
print(f"   Duration: {end_date - last_run_date}")
print(f"\n✅ SLA Processor initialized")
print(f"   Response SLAs: {SLA_CONFIG['response_sla']}")
print(f"   Resolution SLAs: {SLA_CONFIG['resolution_sla']}")

In [ ]:
# Initialize the extractor
extractor = SplunkIncidentExtractor(
    splunk_config=SPLUNK_CONFIG,
    servicenow_config=SERVICENOW_CONFIG,
    chunk_config=CHUNK_CONFIG,
    retry_config=RETRY_CONFIG
)

# Connect to Splunk REST API
connection_success = extractor.connect()
if connection_success:
    print("\n✅ Ready to extract data")
else:
    print("\n❌ Failed to connect. Check your configuration.")

## 5. Execute Extraction

In [ ]:
# Execute the extraction
# Note: Uncomment the following line when ready to run against your Splunk instance

# df_incidents = extractor.extract_all(last_run_date, end_date)

# For demonstration, create a sample DataFrame
print("⚠️ Demonstration mode - uncomment the extraction line above to run against Splunk")
print("\n📋 Example of what the extraction would produce:")

# Sample structure for demonstration
sample_data = {
    '_time': ['2025-01-15 10:30:00', '2025-01-15 11:45:00'],
    'number': ['INC0001234', 'INC0001235'],
    'short_description': ['Password reset request', 'Network connectivity issue'],
    'priority': ['3', '1'],
    'state': ['Resolved', 'In Progress'],
    'assignment_group': ['Service Desk', 'Network Team'],
    'opened_at': ['2025-01-15 10:00:00', '2025-01-15 11:30:00'],
}

df_incidents = pd.DataFrame(sample_data)
print(f"\nSample DataFrame shape: {df_incidents.shape}")
df_incidents.head()

## 6. Save Results & Update Tracking

In [ ]:
def save_to_unity_catalog(df: pd.DataFrame, catalog: str, schema: str, 
                          table: str, mode: str = 'append') -> str:
    """
    Save extraction results to Unity Catalog table.
    
    Args:
        df: Pandas DataFrame with extracted data
        catalog: Unity Catalog name
        schema: Schema/database name
        table: Target table name
        mode: Write mode ('append' or 'overwrite')
        
    Returns:
        Full table name that was written to
    """
    full_table_name = f"{catalog}.{schema}.{table}"
    
    try:
        # Ensure schema exists
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
        
        # Convert pandas DataFrame to Spark DataFrame
        spark_df = spark.createDataFrame(df)
        
        # Add metadata columns
        spark_df = spark_df.withColumn("_extracted_at", current_timestamp())
        
        # Write to Unity Catalog
        spark_df.write \
            .mode(mode) \
            .option("mergeSchema", "true") \
            .saveAsTable(full_table_name)
        
        logger.info(f"💾 Saved to Unity Catalog: {full_table_name}")
        return full_table_name
        
    except Exception as e:
        logger.error(f"Failed to save to Unity Catalog: {e}")
        raise


# =====================================================
# MEDALLION ARCHITECTURE PIPELINE
# =====================================================

if not df_incidents.empty:
    print("=" * 60)
    print("🏗️  MEDALLION ARCHITECTURE PIPELINE")
    print("=" * 60)
    
    # Convert pandas DataFrame to Spark DataFrame for processing
    spark_df = spark.createDataFrame(df_incidents)
    spark_df = spark_df.withColumn("_extracted_at", current_timestamp())
    
    # === BRONZE LAYER: Save Raw Data ===
    print("\n📦 BRONZE LAYER: Saving raw data...")
    bronze_count = save_to_bronze_layer(spark_df, UNITY_CATALOG_CONFIG)
    
    # === SILVER LAYER: Process and Save with SLA Calculations ===
    print("\n⚙️  SILVER LAYER: Processing with SLA calculations...")
    
    # Load the bronze data we just saved (or all bronze data for full reprocessing)
    bronze_df = load_bronze_data_for_processing(UNITY_CATALOG_CONFIG)
    
    if bronze_df is not None and bronze_df.count() > 0:
        # Process through SLA pipeline and save to Silver
        incidents_count, sla_metrics_count = process_and_save_silver_layer(
            bronze_df, 
            sla_processor, 
            UNITY_CATALOG_CONFIG
        )
        
        print("\n" + "=" * 60)
        print("✅ PIPELINE COMPLETE!")
        print("=" * 60)
        print(f"\n📊 Summary:")
        print(f"   Bronze Layer Records: {bronze_count}")
        print(f"   Silver Processed Incidents: {incidents_count}")
        print(f"   Silver SLA Metrics: {sla_metrics_count}")
        print(f"\n📁 Tables Updated:")
        print(f"   Bronze: {UNITY_CATALOG_CONFIG['catalog']}.{UNITY_CATALOG_CONFIG['bronze_schema']}.{UNITY_CATALOG_CONFIG['bronze_incidents_table']}")
        print(f"   Silver Incidents: {UNITY_CATALOG_CONFIG['catalog']}.{UNITY_CATALOG_CONFIG['silver_schema']}.{UNITY_CATALOG_CONFIG['silver_incidents_table']}")
        print(f"   Silver SLA Metrics: {UNITY_CATALOG_CONFIG['catalog']}.{UNITY_CATALOG_CONFIG['silver_schema']}.{UNITY_CATALOG_CONFIG['silver_sla_table']}")
    else:
        print("\n⚠️ No Bronze data available for Silver layer processing")
else:
    print("\n⚠️ No data to save")

In [ ]:
# Update tracking table with extraction results
if not df_incidents.empty:
    tracking.update_tracking(
        job_name=TRACKING_CONFIG['job_name'],
        last_run_date=end_date,
        records_extracted=len(df_incidents),
        status='SUCCESS'
    )
    
    print(f"\n✅ Extraction complete!")
    print(f"   Total records: {len(df_incidents)}")
    print(f"   Tracking updated: {end_date}")
else:
    print("\n⚠️ No records extracted - tracking not updated")

## 7. View Tracking History

In [ ]:
# Display tracking history from Unity Catalog
try:
    tracking_table = f"{UNITY_CATALOG_CONFIG['catalog']}.{UNITY_CATALOG_CONFIG['tracking_schema']}.{UNITY_CATALOG_CONFIG['tracking_table']}"
    df_tracking = spark.sql(f"""
        SELECT * FROM {tracking_table}
        ORDER BY updated_at DESC
        LIMIT 10
    """)
    print("📊 Extraction History:")
    display(df_tracking)
except Exception as e:
    print(f"No tracking history found: {e}")

## 8. View SLA Metrics

In [ ]:
# Display SLA Metrics from Silver Layer
try:
    sla_table = f"{UNITY_CATALOG_CONFIG['catalog']}.{UNITY_CATALOG_CONFIG['silver_schema']}.{UNITY_CATALOG_CONFIG['silver_sla_table']}"
    
    df_sla_metrics = spark.sql(f"""
        SELECT 
            priority,
            total_incidents,
            response_sla_breaches,
            resolution_sla_breaches,
            response_sla_compliance_pct,
            resolution_sla_compliance_pct,
            avg_response_time_minutes,
            avg_resolution_time_minutes,
            open_count,
            resolved_count,
            closed_count,
            _metrics_calculated_at
        FROM {sla_table}
        ORDER BY priority
    """)
    
    print("📊 SLA Metrics by Priority:")
    display(df_sla_metrics)
    
    # Summary stats
    print("\n📈 Overall SLA Summary:")
    summary_df = spark.sql(f"""
        SELECT 
            SUM(total_incidents) as total_incidents,
            SUM(response_sla_breaches) as total_response_breaches,
            SUM(resolution_sla_breaches) as total_resolution_breaches,
            ROUND(AVG(response_sla_compliance_pct), 2) as avg_response_compliance,
            ROUND(AVG(resolution_sla_compliance_pct), 2) as avg_resolution_compliance
        FROM {sla_table}
    """)
    display(summary_df)
    
except Exception as e:
    print(f"No SLA metrics found: {e}")

## 8. Quick Statistics

In [ ]:
# Display quick statistics about the extracted data
print("=" * 60)
print("📊 DATA LAYER STATISTICS")
print("=" * 60)

# Bronze Layer Stats
try:
    bronze_table = f"{UNITY_CATALOG_CONFIG['catalog']}.{UNITY_CATALOG_CONFIG['bronze_schema']}.{UNITY_CATALOG_CONFIG['bronze_incidents_table']}"
    bronze_count = spark.table(bronze_table).count()
    print(f"\n📦 BRONZE LAYER (Raw Data):")
    print(f"   Table: {bronze_table}")
    print(f"   Total Records: {bronze_count}")
except Exception as e:
    print(f"\n📦 Bronze Layer: Not available ({e})")

# Silver Layer Stats
try:
    silver_table = f"{UNITY_CATALOG_CONFIG['catalog']}.{UNITY_CATALOG_CONFIG['silver_schema']}.{UNITY_CATALOG_CONFIG['silver_incidents_table']}"
    silver_df = spark.table(silver_table)
    silver_count = silver_df.count()
    
    print(f"\n⚙️  SILVER LAYER (Processed Data):")
    print(f"   Table: {silver_table}")
    print(f"   Total Unique Incidents: {silver_count}")
    
    # SLA breach summary
    breach_summary = silver_df.select(
        spark_sum(when(col("response_sla_breached") == True, 1).otherwise(0)).alias("response_breaches"),
        spark_sum(when(col("resolution_sla_breached") == True, 1).otherwise(0)).alias("resolution_breaches")
    ).collect()[0]
    
    print(f"   Response SLA Breaches: {breach_summary['response_breaches'] or 0}")
    print(f"   Resolution SLA Breaches: {breach_summary['resolution_breaches'] or 0}")
    
except Exception as e:
    print(f"\n⚙️  Silver Layer: Not available ({e})")

# Pandas DataFrame stats (current extraction)
if not df_incidents.empty:
    print(f"\n📋 CURRENT EXTRACTION:")
    print(f"   Records extracted: {len(df_incidents)}")
    print(f"   Columns: {len(df_incidents.columns)}")
else:
    print("\n📋 No data in current extraction.")

---

## Usage Notes

### First-time Setup:
1. Update `SPLUNK_CONFIG` with your Splunk server details and credentials
2. Update `SERVICENOW_CONFIG` with the correct index and sourcetype for your environment
3. Update `UNITY_CATALOG_CONFIG` with your catalog, schema, and table names
4. Adjust `CHUNK_CONFIG` based on your data volume (larger chunks = fewer queries, smaller chunks = better resilience)
5. Customize `SLA_CONFIG` with your organization's SLA targets by priority

### Medallion Architecture:

**Bronze Layer** (Raw Data):
- Stores all raw records from Splunk exactly as extracted
- Appends new records on each run (maintains full history)
- Schema: `{catalog}.{bronze_schema}.{bronze_incidents_table}`

**Silver Layer** (Processed Data):
- Contains latest state per incident (deduplicated)
- Includes calculated SLA fields:
  - `response_time_minutes`: Time from opened_at to first_response
  - `resolution_time_minutes`: Time from opened_at to resolved_at
  - `response_sla_breached`: Boolean flag for response SLA compliance
  - `resolution_sla_breached`: Boolean flag for resolution SLA compliance
  - `response_sla_remaining_minutes`: Time remaining for response SLA
  - `resolution_sla_remaining_minutes`: Time remaining for resolution SLA
- Schema: `{catalog}.{silver_schema}.{silver_incidents_table}`

**SLA Metrics Table**:
- Aggregated SLA metrics by priority
- Includes compliance percentages, average times, breach counts
- Schema: `{catalog}.{silver_schema}.{silver_sla_table}`

### SLA Configuration:
SLA targets are defined in `SLA_CONFIG` with response and resolution SLAs for each priority level (1-5):
- Priority 1 (Critical): Response 15 min, Resolution 4 hours
- Priority 2 (High): Response 30 min, Resolution 8 hours
- Priority 3 (Medium): Response 1 hour, Resolution 24 hours
- Priority 4 (Low): Response 4 hours, Resolution 72 hours
- Priority 5 (Planning): Response 8 hours, Resolution 1 week

### Customizing the Query:
The `build_query()` method in `SplunkIncidentExtractor` generates the SPL query. Modify the field list to match your ServiceNow incident fields.

### Scheduled Runs:
The tracking table in Unity Catalog automatically maintains the last successful run date, enabling incremental extraction on subsequent runs.

### Prerequisites:
- This notebook must run in a Databricks environment with Unity Catalog enabled
- Ensure you have write permissions to the target catalog and schemas (bronze and silver)